In [1]:
%cd rag-tutorial

/content/rag-tutorial


In [2]:
!pip install openai chromadb python-dotenv

In [3]:
from google.colab import userdata
import os

os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

In [4]:
# TESTING EMBEDDING MODEL
from openai import OpenAI

# Set up the OpenAI client
openai_client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# The OpenAI embedding model we will use
EMBEDDING_MODEL = "text-embedding-3-small"

def get_embedding(text):
    """Generate an embedding for a piece of text using OpenAI."""
    response = openai_client.embeddings.create(
        model=EMBEDDING_MODEL,
        input=text
    )
    return response.data[0].embedding

# Testing that the embedding model does what it's supposed to
if __name__ == "__main__":
    test_embedding = get_embedding("What PSU do I need for the RTX 5080?")
    print(f"Embedding generated successfully.")
    print(f"Dimensions: {len(test_embedding)}")
    print(f"First 5 values: {[round(float(v), 4) for v in test_embedding[:5]]}")


Embedding generated successfully.
Dimensions: 1536
First 5 values: [0.0183, -0.0369, 0.0058, -0.0263, -0.0418]


In [5]:
# TESTING CHUNKING

def chunk_text(text, min_chunk_length=100):
    """
    Split text into chunks by paragraph (double newline).
    Chunks shorter than min_chunk_length characters are discarded,
    as they are unlikely to contain useful information on their own.
    """
    chunks = [chunk.strip() for chunk in text.split("\n\n") if chunk.strip()]
    chunks = [chunk for chunk in chunks if len(chunk) >= min_chunk_length]
    return chunks

# Testing that chunking is working
if __name__ == "__main__":
    sample_text = """This is the first paragraph. It contains some useful information about GPUs and how they affect gaming performance at different resolutions.

This is the second paragraph. It talks about CPUs and motherboards, and why choosing a compatible combination is important for your build.

Short.

This is the fourth paragraph. It has enough content to be worth keeping as a chunk, covering topics like storage options and the difference between NVMe and SATA SSDs."""

    chunks = chunk_text(sample_text)
    print(f"Number of chunks: {len(chunks)}")
    for i, chunk in enumerate(chunks):
        print(f"\nChunk {i+1}: {chunk}")



Number of chunks: 3

Chunk 1: This is the first paragraph. It contains some useful information about GPUs and how they affect gaming performance at different resolutions.

Chunk 2: This is the second paragraph. It talks about CPUs and motherboards, and why choosing a compatible combination is important for your build.

Chunk 3: This is the fourth paragraph. It has enough content to be worth keeping as a chunk, covering topics like storage options and the difference between NVMe and SATA SSDs.


In [6]:
# INGESTING ALL DATA INTO RAG
from openai import OpenAI

#Add these imports below your other imports
import glob
import chromadb

# Neeeded to import FAQs
import json

# Set up the OpenAI client
openai_client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# Add this below your OpenAI setup
# Set up ChromaDB with persistent storage so that the data survives between runs
chroma_client = chromadb.PersistentClient(path="./chroma_db")
collection = chroma_client.get_or_create_collection(name="pc_emporium")

# The OpenAI embedding model we will use
EMBEDDING_MODEL = "text-embedding-3-small"

def get_embedding(text):
    """Generate an embedding for a piece of text using OpenAI."""
    response = openai_client.embeddings.create(
        model=EMBEDDING_MODEL,
        input=text
    )
    return response.data[0].embedding

def chunk_text(text, min_chunk_length=100):
    """
    Split text into chunks by paragraph (double newline).
    Chunks shorter than min_chunk_length characters are discarded,
    as they are unlikely to contain useful information on their own.
    """
    chunks = [chunk.strip() for chunk in text.split("\n\n") if chunk.strip()]
    chunks = [chunk for chunk in chunks if len(chunk) >= min_chunk_length]
    return chunks

def ingest_policies():
    """
    Load policies from individual Markdown files in data/policies/,
    chunk the text, generate an embedding for each chunk, and store in ChromaDB.
    """
    print("\n--- Ingesting policies ---")

    policy_files = glob.glob("data/policies/*.md")

    for filepath in policy_files:
        filename = os.path.basename(filepath)
        slug = filename.replace(".md", "")

        with open(filepath, "r") as f:
            content = f.read()

        lines = content.split("\n")
        title = lines[0].replace("# ", "").strip()
        body = "\n\n".join(lines[1:]).strip()

        chunks = chunk_text(body)
        print(f"  {title}: {len(chunks)} chunk(s)")

        for i, chunk in enumerate(chunks):
            chunk_id = f"policy-{slug}-chunk-{i}"
            embedding = get_embedding(chunk)

            collection.add(
                ids=[chunk_id],
                embeddings=[embedding],
                documents=[chunk],
                metadatas=[{
                    "source": "policy",
                    "slug": slug,
                    "title": title
                }]
            )

    print(f"  Done. {len(policy_files)} policies ingested.")

def ingest_faqs():
    """
    Load FAQs from faqs.json, combine the question and answer into a single
    chunk, generate an embedding, and store in ChromaDB.
    Each FAQ is treated as a single chunk, as they are short by design.
    """
    print("\n--- Ingesting FAQs ---")

    with open("data/faqs.json", "r") as f:
        faqs = json.load(f)

    for faq in faqs:
        # Combine question and answer so the embedding captures both
        text = f"Q: {faq['question']}\nA: {faq['answer']}"
        embedding = get_embedding(text)
        print(f"  {faq['question']}: 1 chunk")

        collection.add(
            ids=[faq["id"]],
            embeddings=[embedding],
            documents=[text],
            metadatas=[{
                "source": "faq",
                "category": faq["category"],
                "question": faq["question"]
            }]
        )

    print(f"  Done. {len(faqs)} FAQs ingested.")

def ingest_blog_posts():
    """
    Load each Markdown file from data/blog-posts/, chunk the body text,
    generate an embedding for each chunk, and store in ChromaDB.
    The title (taken from the first line) and slug are stored as metadata.
    """
    print("\n--- Ingesting blog posts ---")

    blog_files = glob.glob("data/blog-posts/*.md")

    for filepath in blog_files:
        filename = os.path.basename(filepath)
        slug = filename.replace(".md", "")

        with open(filepath, "r") as f:
            content = f.read()

        # The first line of each Markdown file is the title (e.g., # My Title)
        lines = content.split("\n")
        title = lines[0].replace("# ", "").strip()
        body = "\n\n".join(lines[1:]).strip()

        chunks = chunk_text(body)
        print(f"  {title}: {len(chunks)} chunk(s)")

        for i, chunk in enumerate(chunks):
            chunk_id = f"blog-{slug}-chunk-{i}"
            embedding = get_embedding(chunk)

            collection.add(
                ids=[chunk_id],
                embeddings=[embedding],
                documents=[chunk],
                metadatas=[{
                    "source": "blog",
                    "slug": slug,
                    "title": title
                }]
            )

    print(f"  Done. {len(blog_files)} blog posts ingested.")

# Ingesting all data into RAG
if __name__ == "__main__":
    print("Starting ingestion...")
    print(f"Embedding model: {EMBEDDING_MODEL}")

    ingest_policies()
    ingest_faqs()
    ingest_blog_posts()

    total = collection.count()
    print(f"\nIngestion complete. {total} chunks stored in ChromaDB.")



Starting ingestion...
Embedding model: text-embedding-3-small

--- Ingesting policies ---
  Returns Policy: 4 chunk(s)
  Damaged or Lost Parcels Policy: 3 chunk(s)
  Returns Policy (Updated March 2021): 7 chunk(s)
  Price Match Policy: 3 chunk(s)
  Warranty Policy: 2 chunk(s)
  International Shipping Policy: 4 chunk(s)
  Refunds Policy: 3 chunk(s)
  Tax Policy: 3 chunk(s)
  Privacy Policy: 4 chunk(s)
  Dead on Arrival (DOA) Policy: 3 chunk(s)
  Order Cancellation Policy: 3 chunk(s)
  Account Termination Policy: 3 chunk(s)
  Pre-Order Policy: 3 chunk(s)
  Financing Policy: 3 chunk(s)
  Backorder Policy: 3 chunk(s)
  Recycling & Disposal Policy: 3 chunk(s)
  Technical Support Policy: 3 chunk(s)
  Repair & Servicing Policy: 3 chunk(s)
  Payment Methods Policy: 3 chunk(s)
  Domestic Shipping Policy: 3 chunk(s)
  Done. 20 policies ingested.

--- Ingesting FAQs ---
  Is the AMD Ryzen 9 9950X compatible with DDR4 RAM?: 1 chunk
  Is the AMD Ryzen 7 9700X compatible with DDR4 RAM?: 1 chunk
  Ca

In [7]:
# Sanity check - check_vectordb.py
import chromadb

chroma_client = chromadb.PersistentClient(path="./chroma_db")
collection = chroma_client.get_or_create_collection(name="pc_emporium")

# ---- Total chunk count ----
print("=== Total chunks in ChromaDB ===")
print(f"{collection.count()} chunks stored")
print()

# ---- Chunks per source ----
print("=== Chunks by source ===")
for source in ["policy", "faq", "blog"]:
    results = collection.get(where={"source": source})
    print(f"  {source}: {len(results['ids'])} chunks")
print()

# ---- Look at a sample chunk ----
print("=== Sample chunk ===")
results = collection.get(
    ids=["policy-returns-policy-2026-chunk-0"],
    include=["embeddings", "documents"]
)

print(f"Text content (first 300 characters):")
print(f"  {results['documents'][0][:300]}...")
print()

print(f"Vector embedding (a vector representation of the text above):")
embedding = results['embeddings'][0]
print(f"  Dimensions: {len(embedding)} numbers")
print(f"  First 10 values: {[round(float(v), 4) for v in embedding[:10]]}")



=== Total chunks in ChromaDB ===
162 chunks stored

=== Chunks by source ===
  policy: 66 chunks
  faq: 25 chunks
  blog: 71 chunks

=== Sample chunk ===
Text content (first 300 characters):
  PC Emporium accepts returns within 30 days of delivery for most products. Items must be returned in their original packaging and in an unused, resalable condition....

Vector embedding (a vector representation of the text above):
  Dimensions: 1536 numbers
  First 10 values: [-0.0058, 0.0415, 0.0553, 0.0426, 0.0235, 0.0098, -0.0103, 0.0721, 0.0417, 0.0544]


In [9]:
# TESTING RETRIEVAL
import os
from dotenv import load_dotenv
from openai import OpenAI
import chromadb

# Load environment variables from .env file
load_dotenv()

# Set up the OpenAI client
openai_client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# Connect to the existing ChromaDB database created by ingest.py
chroma_client = chromadb.PersistentClient(path="./chroma_db")
collection = chroma_client.get_or_create_collection(name="pc_emporium")

EMBEDDING_MODEL = "text-embedding-3-small"
# The number of results the retrieval phase should return
N_RESULTS = 5


# This is the same as get_embedding in ingest.py
def get_embedding(text):
    """Generate an embedding for a piece of text using OpenAI."""
    response = openai_client.embeddings.create(
        model=EMBEDDING_MODEL,
        input=text
    )
    return response.data[0].embedding


def retrieve(question):
    """
    Convert the question to an embedding and query ChromaDB
    to find the most semantically similar chunks.
    """
    question_embedding = get_embedding(question)

    results = collection.query(
        query_embeddings=[question_embedding],
        n_results=N_RESULTS,
        include=["documents", "metadatas", "distances"]
    )

    return results

# Testing the retrieval mechanism
if __name__ == "__main__":
    results = retrieve("What is the returns policy?")
    chunks = results["documents"][0]
    metadatas = results["metadatas"][0]
    # Distance = how similar each chunk is to the original question.
    # A lower distance = more similar
    distances = results["distances"][0]

    for i, (chunk, metadata, distance) in enumerate(zip(chunks, metadatas, distances)):
        print(f"\n[{i+1}] Source: {metadata['source']} | Title: {metadata.get('title', 'N/A')} | Distance: {round(distance, 4)}")
        print(f"     {chunk[:150]}...")



[1] Source: policy | Title: Returns Policy (Updated March 2021) | Distance: 0.8921
     To initiate a return, email support@pcemporium.com with your order number and a brief description of the reason for your return. No further documentat...

[2] Source: policy | Title: Returns Policy | Distance: 0.9154
     To initiate a return, customers must contact support@pcemporium.com with their order number and reason for return. Once the return is approved, custom...

[3] Source: policy | Title: Order Cancellation Policy | Distance: 0.9568
     If the order has already been dispatched, it cannot be cancelled but can be returned under the standard returns policy once delivered....

[4] Source: policy | Title: Returns Policy | Distance: 0.9597
     PC Emporium accepts returns within 30 days of delivery for most products. Items must be returned in their original packaging and in an unused, resalab...

[5] Source: policy | Title: Returns Policy (Updated March 2021) | Distance: 1.0237
     We have

In [10]:
# QUERYING THE FULL RAG
import os
from dotenv import load_dotenv
from openai import OpenAI
import chromadb

# Load environment variables from .env file
load_dotenv()

# Set up the OpenAI client
openai_client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# Connect to the existing ChromaDB database created by ingest.py
chroma_client = chromadb.PersistentClient(path="./chroma_db")
collection = chroma_client.get_or_create_collection(name="pc_emporium")

EMBEDDING_MODEL = "text-embedding-3-small"
# The number of results the retrieval phase should return
N_RESULTS = 5

# Define the model you'll be using and the specific system prompt that helps it handle the RAG data.
CHAT_MODEL = "gpt-4o-mini"

SYSTEM_PROMPT = """You are a helpful assistant for PC Emporium, a PC components retailer.
Answer the user's question using only the context provided below.
If the context does not contain enough information to answer the question, say:
"I'm sorry, I don't have enough information to answer that question."
Do not make up information that is not in the context."""

# This is the same as get_embedding in ingest.py
def get_embedding(text):
    """Generate an embedding for a piece of text using OpenAI."""
    response = openai_client.embeddings.create(
        model=EMBEDDING_MODEL,
        input=text
    )
    return response.data[0].embedding


def retrieve(question):
    """
    Convert the question to an embedding and query ChromaDB
    to find the most semantically similar chunks.
    """
    question_embedding = get_embedding(question)

    results = collection.query(
        query_embeddings=[question_embedding],
        n_results=N_RESULTS,
        include=["documents", "metadatas", "distances"]
    )

    return results

# Testing the retrieval mechanism
def generate(question, context_chunks):
    """
    Pass the question and retrieved context chunks to the LLM
    and return its answer.
    """
    # Format the retrieved chunks into a single context string
    context = "\n\n---\n\n".join(context_chunks)

    response = openai_client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"}
        ]
    )

    return response.choices[0].message.content


def ask(question):
    """
    Full RAG pipeline: Retrieve relevant chunks, then generate an answer.
    """
    print(f"\nQuestion: {question}")
    print("\nSearching knowledge base...")

    # Step 1: Retrieve the most relevant chunks from ChromaDB
    results = retrieve(question)
    chunks = results["documents"][0]
    metadatas = results["metadatas"][0]
    distances = results["distances"][0]

    # Show the reader what was retrieved (for transparency)
    print(f"\nTop {N_RESULTS} most relevant chunks found:")
    for i, (chunk, metadata, distance) in enumerate(zip(chunks, metadatas, distances)):
        print(f"\n  [{i+1}] Source: {metadata['source']} | Title: {metadata.get('title', metadata.get('question', 'N/A'))} | Distance: {round(distance, 4)}")
        print(f"       {chunk[:150]}...")

    # Step 2: Generate an answer using the retrieved chunks as context
    print("\nGenerating answer...")
    answer = generate(question, chunks)

    print(f"\nAnswer: {answer}")
    return answer

if __name__ == "__main__":
    print("PC Emporium RAG — type your question or 'quit' to exit.")
    print("=" * 60)

    while True:
        question = input("\nYour question: ").strip()

        if question.lower() in ["quit", "exit", "q"]:
            print("Goodbye!")
            break

        if not question:
            continue

        ask(question)


PC Emporium RAG — type your question or 'quit' to exit.

Your question: Is the Ryzen 9 9950X compatible with DDR4 RAM?

Question: Is the Ryzen 9 9950X compatible with DDR4 RAM?

Searching knowledge base...

Top 5 most relevant chunks found:

  [1] Source: faq | Title: Is the AMD Ryzen 9 9950X compatible with DDR4 RAM? | Distance: 0.2724
       Q: Is the AMD Ryzen 9 9950X compatible with DDR4 RAM?
A: No. The Ryzen 9 9950X uses the AM5 socket, which only supports DDR5 RAM. DDR4 RAM is not comp...

  [2] Source: faq | Title: Is the AMD Ryzen 7 9700X compatible with DDR4 RAM? | Distance: 0.489
       Q: Is the AMD Ryzen 7 9700X compatible with DDR4 RAM?
A: No. Like all Ryzen 9000 series CPUs, the Ryzen 7 9700X uses the AM5 socket which only support...

  [3] Source: faq | Title: What is the correct RAM speed for the AMD Ryzen 9000 series? | Distance: 0.7031
       Q: What is the correct RAM speed for the AMD Ryzen 9000 series?
A: AMD recommends DDR5-6000 as the sweet spot for Ryzen 9000 se

KeyboardInterrupt: Interrupted by user